# Caracal Base 3B · Continued Pretrain · Colab Pro+

Plug-and-play. Mesmo que Kaggle mas Colab Pro+ pode pegar A100 esporadico (rodar 2-3x mais rapido).

**Antes:**
1. Garantir Pro+ ativo + Runtime > Change runtime > A100 (se disponivel)
2. Verificar SCHEDULE.md
3. HF_TOKEN e WANDB_API_KEY como Colab Secrets (Files > Secrets sidebar)

In [ ]:
SESSION_NUMBER = 1
RESUME_REVISION = None
OUTPUT_REVISION = 'step-5500'
STEPS_TO_RUN = 5500
FOUNDER_HANDLE = 'PAMF2'

In [ ]:
!nvidia-smi --query-gpu=name,memory.free --format=csv,noheader
!git clone https://github.com/iterate-labs-ai/caracal-1.git
%cd caracal-1
!git checkout dev
!pip install --quiet 'unsloth @ git+https://github.com/unslothai/unsloth.git'
!pip install --quiet trl peft bitsandbytes datasets transformers accelerate huggingface_hub wandb pyyaml sentence-transformers

In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
!huggingface-cli login --token $HF_TOKEN --add-to-git-credential
!wandb login $WANDB_API_KEY

In [ ]:
!bash scripts/preflight.sh

In [ ]:
import subprocess, sys
resume_arg = []
if RESUME_REVISION:
    resume_arg = ['--resume-from', f'huggingface://iterate-labs/caracal-base-pretrain@{RESUME_REVISION}']
cmd = [sys.executable, 'train/continued_pretrain.py',
       '--config', 'train/configs/caracal_base_3b.yaml',
       *resume_arg,
       '--steps-to-run', str(STEPS_TO_RUN),
       '--output', './ckpt-out',
       '--hf-revision-out', OUTPUT_REVISION]
subprocess.run(cmd, check=True)